# Weather Condition Classification from Outdoor Photographs
### MSML640 Final Project: Akhil Shekkari & Siddharth Pathania

**Task:** Five-class image classification (clear, cloudy, foggy, rainy, snowy)  
**Backbone:** EfficientNet-B0 (ImageNet pretrained, ~5.3 M parameters)  
**Experiment:** Four training configurations studied with ablation across augmentation and synthetic data.

In [ ]:
# Run once in Colab to install dependencies
# !pip install timm torch torchvision seaborn scikit-learn tqdm diffusers transformers accelerate -q

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

# --- path setup -----------------------------------------------------------
# Adjust REPO_ROOT if running from a different working directory
REPO_ROOT = Path('.').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

DATA_ROOT   = str(REPO_ROOT / 'data')
RESULTS_DIR = str(REPO_ROOT / 'results')
BACKBONE    = 'efficientnet_b0'
EPOCHS      = 30
BATCH_SIZE  = 32

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Repo root: {REPO_ROOT}')

## 1. Data Preparation

**Before running this notebook:**
1. Add seed-pool images to `data/raw/<class>/`
2. Add self-captured images to `data/captured/<class>/`
3. (Optional) Run `src/synthesize.py` to populate `data/synthetic/<class>/`
4. Run the cell below to create train / val / test split files.

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'src/prepare_data.py', '--data-root', DATA_ROOT],
    capture_output=True, text=True, cwd=str(REPO_ROOT)
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

### Class distribution

In [ ]:
from collections import defaultdict

CLASSES = ['clear', 'cloudy', 'foggy', 'rainy', 'snowy']
splits  = ['train', 'val', 'test']
counts  = {}

for split in splits:
    p = Path(DATA_ROOT) / 'splits' / f'{split}.json'
    if not p.exists():
        continue
    with open(p) as f:
        entries = json.load(f)
    by_cls = defaultdict(int)
    for e in entries:
        if not e.get('synthetic', False):
            by_cls[e['label']] += 1
    counts[split] = [by_cls[c] for c in CLASSES]

x = np.arange(len(CLASSES))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 4))
for i, (split, clr) in enumerate(zip(splits, ['steelblue', 'orange', 'green'])):
    if split in counts:
        ax.bar(x + i * width, counts[split], width, label=split, color=clr)
ax.set_xticks(x + width); ax.set_xticklabels(CLASSES)
ax.set_ylabel('Image count'); ax.set_title('Class distribution per split (real images only)')
ax.legend(); plt.tight_layout(); plt.show()

## 2. Training: All Four Configurations

| Config | Data | Augmentation |
|--------|------|--------------|
| 1 | Our data only | None |
| 2 | Our data | Flip, crop, color jitter, rotation |
| 3 | Our data + Stable Diffusion | None |
| 4 | Our data + Stable Diffusion | Same as Config 2 |

In [ ]:
from train import train
import argparse

def make_args(config):
    a = argparse.Namespace()
    a.config      = config
    a.data_root   = DATA_ROOT
    a.results_dir = RESULTS_DIR
    a.backbone    = BACKBONE
    a.epochs      = EPOCHS
    a.batch_size  = BATCH_SIZE
    a.lr          = 3e-4
    a.num_workers = 2
    return a

all_histories = {}
for cfg in [1, 2, 3, 4]:
    print(f'\n{"="*60}')
    print(f'CONFIG {cfg}')
    print('='*60)
    history, best_val = train(make_args(cfg))
    all_histories[cfg] = history
    print(f'Config {cfg} done. Best val acc: {best_val:.4f}')

## 3. Learning Curves: All Configs

In [ ]:
from utils import plot_all_val_curves

out = Path(RESULTS_DIR) / 'all_val_curves.png'
plot_all_val_curves(RESULTS_DIR, [1, 2, 3, 4], out)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ['steelblue', 'orange', 'green', 'red']
for cfg, clr in zip([1, 2, 3, 4], colors):
    if cfg not in all_histories:
        continue
    h = all_histories[cfg]
    ep = range(1, len(h['val_loss']) + 1)
    ax1.plot(ep, h['val_loss'], label=f'Config {cfg}', color=clr)
    ax2.plot(ep, h['val_acc'],  label=f'Config {cfg}', color=clr)

for ax, title, ylabel in [
    (ax1, 'Validation Loss',     'Loss'),
    (ax2, 'Validation Accuracy', 'Accuracy'),
]:
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.set_title(title); ax.legend(); ax.grid(True)

plt.tight_layout(); plt.show()

## 4. Test-Set Evaluation

In [ ]:
from evaluate import evaluate_config
from utils import compare_confusion_matrices
import matplotlib.image as mpimg

all_cms, done_cfgs = [], []
for cfg in [1, 2, 3, 4]:
    ckpt = Path(RESULTS_DIR) / f'config{cfg}' / 'best_model.pt'
    if not ckpt.exists():
        print(f'Config {cfg}: no checkpoint, skipping')
        continue
    cm, report = evaluate_config(cfg, DATA_ROOT, RESULTS_DIR, BACKBONE, BATCH_SIZE)
    all_cms.append(cm)
    done_cfgs.append(cfg)

if len(all_cms) > 1:
    cmp_path = Path(RESULTS_DIR) / 'confusion_matrix_comparison.png'
    compare_confusion_matrices(all_cms, [f'Config {c}' for c in done_cfgs], CLASSES, cmp_path)
    img = mpimg.imread(cmp_path)
    plt.figure(figsize=(20, 5)); plt.imshow(img); plt.axis('off')
    plt.title('Confusion matrices - test set (row-normalised)'); plt.show()

## 5. Robustness Suite

Four synthetic perturbations are applied to the clean test set and accuracy is measured per-config.

In [ ]:
from robustness import run_robustness_suite
import argparse

rob_args = argparse.Namespace(
    configs     = done_cfgs,
    data_root   = DATA_ROOT,
    results_dir = RESULTS_DIR,
    backbone    = BACKBONE,
    batch_size  = BATCH_SIZE,
)
rob_results = run_robustness_suite(rob_args)

In [ ]:
import matplotlib.image as mpimg
rob_plot = Path(RESULTS_DIR) / 'robustness_plot.png'
if rob_plot.exists():
    img = mpimg.imread(rob_plot)
    plt.figure(figsize=(12, 5)); plt.imshow(img); plt.axis('off')
    plt.title('Robustness Suite - Accuracy per Perturbation'); plt.show()

## 6. Per-class Error Analysis

Which classes are hardest to classify, and which misclassification pairs are most common?

In [ ]:
import pandas as pd

rows = []
for cfg in done_cfgs:
    rp_path = Path(RESULTS_DIR) / f'config{cfg}' / 'classification_report.json'
    if not rp_path.exists():
        continue
    with open(rp_path) as f:
        rp = json.load(f)
    for cls in CLASSES:
        if cls in rp:
            rows.append({
                'Config':    cfg,
                'Class':     cls,
                'Precision': round(rp[cls]['precision'], 3),
                'Recall':    round(rp[cls]['recall'],    3),
                'F1':        round(rp[cls]['f1-score'],  3),
                'Support':   rp[cls]['support'],
            })

df = pd.DataFrame(rows)
if not df.empty:
    pivot = df.pivot_table(index='Class', columns='Config', values='F1')
    print('F1-score per class across configs:')
    display(pivot.style.background_gradient(cmap='RdYlGn', axis=None))

## 7. Summary & Hypothesis Assessment

**Central hypothesis:**  
> Augmentation will improve robustness most on perturbations that resemble the augmentation distribution (color jitter vs. overcast, blur vs. rain), while synthesis will help most on minority classes where real samples are scarce.

Fill in your observations below after running all experiments.

In [ ]:
# Tabulate final test accuracy across configs
summary_rows = []
for cfg in done_cfgs:
    rp_path = Path(RESULTS_DIR) / f'config{cfg}' / 'classification_report.json'
    if rp_path.exists():
        with open(rp_path) as f:
            rp = json.load(f)
        summary_rows.append({'Config': cfg, 'Test Accuracy': round(rp['accuracy'], 4)})

if summary_rows:
    df_sum = pd.DataFrame(summary_rows).set_index('Config')
    display(df_sum.style.highlight_max(color='lightgreen'))

# Robustness delta: Config 2 vs Config 1 (augmentation effect on robustness)
if rob_results and 1 in done_cfgs and 2 in done_cfgs:
    print('\nRobustness delta (Config 2 - Config 1):')
    for pert, cfg_accs in rob_results.items():
        delta = cfg_accs.get(2, 0) - cfg_accs.get(1, 0)
        print(f'  {pert:<26}: {delta:+.4f}')